In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io
import re

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 25
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## Yale EPI Pipeline

**Source:** Yale Environmental Performance Index
**Access:** Automated — scrapes all pages of downloads page, auto-detects all available editions
**Download instructions:** See `docs/instructions_data_maintenance.md` — YALE_EPI section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| EPI sub-components (policy/institutional) | Environmental/climate governance | Primary tier 1 |

### Note
Master PDF specifies policy and institutional sub-components, not headline composite.
Specific sub-components to be selected at metric pass.
All available biennial editions stacked automatically — years derived from filenames.

In [19]:
import requests
import io
import re
import pandas as pd
from datetime import datetime

EPI_BASE = "https://epi.yale.edu"

# Sub-indices to retain — those consistently available across editions
# 2024-only indices kept but will be NaN for earlier editions
CONSISTENT_INDICES = ['EPI', 'CCH', 'ECO', 'HLT', 'BDH', 'AGR', 'FSH', 'WRS']
EXTENDED_INDICES   = ['MKP', 'MPE', 'MHP']  # newer editions only
ALL_INDICES        = CONSISTENT_INDICES + EXTENDED_INDICES

def get_all_epi_result_urls():
    """
    Scrape all pages of epi-downloads to find all available edition results CSVs.
    Returns list of (year, url) tuples sorted oldest to newest.
    """
    results = []
    page_num = 0
    while True:
        url = f"{EPI_BASE}/epi-downloads?page={page_num}" if page_num > 0 else f"{EPI_BASE}/epi-downloads"
        page = requests.get(url, headers=BROWSER_HEADERS, timeout=30)
        links = re.findall(r'(https://epi\.yale\.edu/downloads/epi(\d{4})results[^"\'<>\s]*\.csv)', page.text)
        if not links:
            break
        for full_url, year in links:
            results.append((int(year), full_url))
        # Check if there are more pages
        if f'page={page_num + 1}' not in page.text:
            break
        page_num += 1
    # Deduplicate and sort
    results = sorted(set(results), key=lambda x: x[0])
    print(f"Found {len(results)} EPI editions: {[y for y,_ in results]}")
    return results

def extract_epi_edition(df, year, indices):
    """Extract selected sub-indices from one EPI edition."""
    id_cols = ['code', 'iso', 'country']
    new_cols = [f"{idx}.new" for idx in indices if f"{idx}.new" in df.columns]
    out = df[id_cols + new_cols].copy()
    out = out.rename(columns={
        'code':    'country_code_epi',
        'iso':     'country_code',
        'country': 'country_name',
    })
    # Rename index columns with epi_ prefix
    rename_map = {f"{idx}.new": f"epi_{idx.lower()}" for idx in indices if f"{idx}.new" in df.columns}
    out = out.rename(columns=rename_map)
    out['year'] = year
    return out

# Discover and download all available editions
edition_urls = get_all_epi_result_urls()
latest_year = max(y for y,_ in edition_urls)

frames = []
for year, url in edition_urls:
    print(f"Downloading EPI {year}...")
    r = requests.get(url, headers=BROWSER_HEADERS, timeout=30)
    df = pd.read_csv(io.StringIO(r.text))
    frame = extract_epi_edition(df, year, ALL_INDICES)
    frames.append(frame)
    print(f"  Shape: {frame.shape}")

# Stack all editions
epi = pd.concat(frames, ignore_index=True)
epi = epi.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"\nCombined shape: {epi.shape}")
print(f"Years: {sorted(epi['year'].unique())}")
print(f"Countries: {epi['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (epi.isnull().sum() / len(epi) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Found 2 EPI editions: [2022, 2024]
  Shape: (180, 12)
  Shape: (180, 15)

Combined shape: (360, 15)
Years: [np.int64(2022), np.int64(2024)]
Countries: 181

Missing values (%):
epi_mpe    63.6
epi_mkp    63.3
epi_mhp    60.0
epi_fsh    23.1
epi_wrs     0.8
dtype: float64


In [20]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "epi_clean.csv")
epi.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {epi.shape}")

# Derive data currency from data — no hardcoding
data_as_of_date = str(latest_year)
editions_found = sorted(epi['year'].unique().tolist())

# Update download log
update_entry(
    "YALE_EPI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="epi_clean.csv",
    latest_available_version=data_as_of_date,
    notes=f"All available editions auto-detected and stacked: {editions_found}. "
          f"Consistent sub-indices: {CONSISTENT_INDICES}. "
          f"Extended (newer editions only): {EXTENDED_INDICES}. "
          f"Methodology differs between editions — limited comparability. "
          f"Master PDF: use policy/institutional sub-components selectively — metric pass pending. "
          f"Coverage: ~180 countries, biennial."
)
print_entry("YALE_EPI")

Written: /Users/boulanger/Documents/governance-framework/data/processed/epi_clean.csv
Shape: (360, 15)
[download_log] Updated entry for YALE_EPI
  source_id: YALE_EPI
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2024
  local_filename: epi_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: All available editions auto-detected and stacked: [2022, 2024]. Consistent sub-indices: ['EPI', 'CCH', 'ECO', 'HLT', 'BDH', 'AGR', 'FSH', 'WRS']. Extended (newer editions only): ['MKP', 'MPE', 'MHP']. Methodology differs between editions — limited comparability. Master PDF: use policy/institutional sub-components selectively — metric pass pending. Coverage: ~180 countries, biennial.
